In [1]:
# 运行这个单元格安装依赖（大约需要 1~2 分钟）
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 21.3 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.7 MB/s eta 0:00:0000:0100:01m
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for xformers
  Running setup.py clean for xformers
Failed to build xformers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (xformers)
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-9lves2ww/unsloth_dfb5a9d53f3b4b818650a3c821ab28eb
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-9lves2ww/unsloth_dfb5a9d5

In [2]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # 最大上下文长度
dtype = None # 自动检测
load_in_4bit = True # 开启 4bit，显存立省 70%

# 加载 Gemma 4 E2B 对话模型
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 配置 LoRA 微调参数（只训练一小部分权重）
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj","gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.3: Fast Gemma4 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma4 won't work! Using float32.


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


In [3]:
from datasets import load_dataset, concatenate_datasets

# 1. 下载通用中文指令数据 (Alpaca-zh, 2000条，保证日常对话和基础情商)
print("正在下载通用中文指令数据 (Alpaca-zh, 2000条)...")
alpaca_ds = load_dataset("shibing624/alpaca-zh", split="train[:2000]")

# 2. 下载 DeepSeek-R1 官方蒸馏中文深度思维链数据 (1000条，注入深度慢思考灵魂)
print("正在下载 DeepSeek-R1 中文深度思维链数据...")
r1_ds = load_dataset("Congliu/Chinese-DeepSeek-R1-Distill-data-110k-SFT", split="train[:1500]")

# 过滤超长文本（保留 2000 字符内，保证 T4 显存绝对安全）
r1_ds = r1_ds.filter(lambda x: len(x.get("output", "")) <= 2000 and len(x.get("output", "")) >= 80)
r1_ds = r1_ds.select(range(min(1000, len(r1_ds))))

# 3. 转换为 Gemma 官方标准模板
def format_alpaca(batch):
    texts = []
    for inst, inp, out in zip(batch["instruction"], batch["input"], batch["output"]):
        user_msg = f"{inst}\n{inp}".strip() if inp else inst
        text = f"<start_of_turn>user\n{user_msg}<end_of_turn>\n<start_of_turn>model\n{out}<end_of_turn>"
        texts.append(text)
    return {"text": texts}

def format_r1(batch):
    texts = []
    for inst, inp, out in zip(batch["instruction"], batch["input"], batch["output"]):
        user_msg = f"{inst}\n{inp}".strip() if inp else inst
        # out 已自带的 <think>...</think> 思考过程和结论
        text = f"<start_of_turn>user\n{user_msg}<end_of_turn>\n<start_of_turn>model\n{out}<end_of_turn>"
        texts.append(text)
    return {"text": texts}

alpaca_formatted = alpaca_ds.map(format_alpaca, batched=True, remove_columns=alpaca_ds.column_names)
r1_formatted = r1_ds.map(format_r1, batched=True, remove_columns=r1_ds.column_names)

# 4. 混合拼接（3000条：文武双全，既懂通用对话，又懂 R1 深度推演）
dataset = concatenate_datasets([alpaca_formatted, r1_formatted]).shuffle(seed=3407)

print(f"\n混合数据集准备完毕！总数据量: {len(dataset)} 条 (2000 通用 + 1000 R1 思维链)")
print("展示一条 R1 样本片段:\n", [x for x in dataset["text"] if "<think>" in x][0][:300], "...")

正在下载通用中文指令数据 (Alpaca-zh, 2000条)...


README.md:   0%|          | 0.00/1.65k [00:00<?, ?B/s]

alpaca_gpt4_data_zh.json: reconstructing file:   0%|          |  0.00B / 35.1MB            

alpaca_gpt4_data_zh.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/48818 [00:00<?, ? examples/s]

...

Generating train split:   0%|          | 0/110000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/778 [00:00<?, ? examples/s]


混合数据集准备完毕！总数据量: 2778 条 (2000 通用 + 1000 R1 思维链)
展示一条 R1 样本片段:
 <start_of_turn>user
当黎明的第一缕阳光穿透密集的树冠，森林中回荡着阵阵悦耳的鸟鸣，那场景真是令人心旷神怡。<end_of_turn>
<start_of_turn>model
<think>
好的，用户给了一个关于黎明森林的场景描述，需要我来扩展和润色。首先，我得仔细分析原文，看看有哪些地方可以加强。原文提到黎明的阳光穿透树冠，鸟鸣声回荡，让人心旷神怡。用户可能希望这段文字更加生动、细腻，增强画面感和情感共鸣。

接下来，我需要考虑如何扩展。视觉方面，可以详细描写阳光的颜色变化，比如从暖红到金橙，树冠的形态，如羽状复叶、蕨类植物。听觉方面，鸟鸣声可以分层次，比如画眉的清脆、 ...


In [4]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 1, # 跑满 1 个 Epoch（约 375 步）
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        output_dir = "outputs",
    ),
)

# 开始训练
trainer.train()

Unsloth: Switching to float32 training since model cannot work with float16
Unsloth: transformers renamed `push_to_hub_token` to `hub_token`. Forwarding your value to `hub_token` - update your code when convenient. If you also passed `hub_token` as None, that is its default here and cannot be distinguished from leaving it unset, so `push_to_hub_token` was used; drop `push_to_hub_token` to keep it.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2778 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,778 | Num Epochs = 1 | Total steps = 348
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,158,208 of 5,128,455,712 (0.47% trained)


Step,Training Loss
10,3.718994
20,2.372565
30,1.886133
40,1.657900
50,1.600730
60,1.647222
70,1.565434
80,1.476911
90,1.454625
100,1.665691


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-348/tokenizer_config.json.


TrainOutput(global_step=348, training_loss=1.6059939285804485, metrics={'train_runtime': 2734.4433, 'train_samples_per_second': 1.016, 'train_steps_per_second': 0.127, 'total_flos': 2.215314977351731e+16, 'train_loss': 1.6059939285804485, 'epoch': 1.0})

In [ ]:
# 1. 切换到推理加速模式
FastLanguageModel.for_inference(model)

# 2. 准备测试问题列表（包含经典的思维链陷阱题）
test_questions = [
    "9.11 和 9.9 哪个数字更大？",
]

print("🤖 开始检验 Gemma 4 + DeepSeek-R1 灵魂学习成果：\n" + "="*50)

for idx, q in enumerate(test_questions, 1):
    prompt = f"<start_of_turn>user\n{q}<end_of_turn>\n<start_of_turn>model\n"
    inputs = tokenizer(text = [prompt], return_tensors = "pt").to("cuda")
    
    # 留出充足草稿空间（max_new_tokens=512）
    outputs = model.generate(
        **inputs, 
        max_new_tokens = 2048, 
        use_cache = True,
        temperature = 0.6,
    )
    
    reply = tokenizer.batch_decode(outputs)[0]
    answer = reply.split("<start_of_turn>model\n")[-1].replace("<end_of_turn>", "")
    
    print(f"\n【问题 {idx}】：{q}")
    print(f"【Gemma 4 (R1-Distill) 回答】：\n{answer.strip()}")
    print("-" * 50)

🤖 开始检验 Gemma 4 + DeepSeek-R1 灵魂学习成果：

【问题 1】：9.11 和 9.9 哪个数字更大？
【Gemma 4 (R1-Distill) 回答】：
比较9.11和9.9：

1. 整数部分相同：两者都是9，因此整数
--------------------------------------------------


In [ ]:
import shutil
from google.colab import files

# 保存并打包专属的 Gemma-4-DeepSeek-R1-Distill 适配器权重
print("正在保存专属 R1 思维链 LoRA 适配器...")
model.save_pretrained("my_gemma_r1_lora")
tokenizer.save_pretrained("my_gemma_r1_lora")

print("正在打包为 zip 文件...")
shutil.make_archive("my_gemma_r1_lora", 'zip', "my_gemma_r1_lora")

print("打包成功，开始触发下载！")
files.download("my_gemma_r1_lora.zip")